<a href="https://colab.research.google.com/github/fareyahossain835-bit/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
print("""The baseline rule ranks items that may need attention using simple observed signals.

The score increases when:
- the item is stale and has not been updated recently
- recent impressions have decreased compared with the previous period

Items with higher scores are placed higher in the review queue.

This rule is used only for decision-support. It does not predict future performance.

Reason codes:

- NEEDS_REVIEW: The item triggered one or more warning signals and should be reviewed.
- MONITOR: No strong warning signal was observed.""")

The baseline rule ranks items that may need attention using simple observed signals.

The score increases when:
- the item is stale and has not been updated recently
- recent impressions have decreased compared with the previous period

Items with higher scores are placed higher in the review queue.

This rule is used only for decision-support. It does not predict future performance.

Reason codes:

- NEEDS_REVIEW: The item triggered one or more warning signals and should be reviewed.
- MONITOR: No strong warning signal was observed.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

Paste your Hugging Face READ token: ··········


In [2]:
import pandas as pd
import numpy as np
import os


# Load performance data

performance = con.sql(f"""
SELECT *
FROM '{REL}/fact_content_daily_performance_sample.parquet'
""").df()



# Load content data

content = con.sql(f"""
SELECT *
FROM '{REL}/dim_content.parquet'
""").df()



print("Performance shape:", performance.shape)
print("Content shape:", content.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Performance shape: (11694072, 31)
Content shape: (519606, 26)


In [3]:
df = con.sql(f"""
SELECT
    p.*,
    c.content_updated_date,
    c.content_created_date
FROM '{REL}/fact_content_daily_performance_sample.parquet' p
LEFT JOIN '{REL}/dim_content.parquet' c
ON p.content_hash_id = c.content_hash_id
LIMIT 200000
""").df()

print(df.shape)

df.head()

(200000, 33)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,content_updated_date,content_created_date
0,2026-06-02,client_f623b01661d4bfe4,content_575a1ed6e670a9a3,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,2026-06,2026-05-20,2025-08-07
1,2026-06-02,client_f623b01661d4bfe4,content_22290552590d7529,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,2026-06,2026-05-20,2025-08-07
2,2026-06-02,client_f623b01661d4bfe4,content_bc7ee2e19eb9ce58,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,2026-06,2026-05-20,2025-08-07
3,2026-06-02,client_f623b01661d4bfe4,content_08bfdd6c0a86993f,True,True,True,False,6,0,35,...,0,0,0,0,0,0,0,2026-06,2026-05-20,2025-08-07
4,2026-06-02,client_f623b01661d4bfe4,content_4819593b7027a1c2,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,2026-06,2026-05-20,2025-08-07


In [4]:
import numpy as np
import pandas as pd

df["content_updated_date"] = pd.to_datetime(
    df["content_updated_date"]
)

today = pd.to_datetime(df["report_date"]).max()

df["days_since_last_update"] = (
    today - df["content_updated_date"]
).dt.days


df[["content_updated_date", "days_since_last_update"]].head()

,content_updated_date,days_since_last_update
0,2026-05-20,14
1,2026-05-20,14
2,2026-05-20,14
3,2026-05-20,14
4,2026-05-20,14


In [5]:
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)


df[["gsc_impressions", "gsc_clicks", "ctr"]].head()

,gsc_impressions,gsc_clicks,ctr
0,0,0,0.0
1,0,0,0.0
2,0,0,0.0
3,6,0,0.0
4,0,0,0.0


In [6]:
df["baseline_score"] = 0


# Stale content signal

df.loc[
    df["days_since_last_update"] >= 180,
    "baseline_score"
] += 50



# Low CTR signal

df.loc[
    (df["gsc_impressions"] > 100) &
    (df["ctr"] < 0.02),
    "baseline_score"
] += 50


df["baseline_score"].value_counts()

,count
baseline_score,
0,189243
50,10757


In [7]:
df["reason_code"] = np.where(
    df["baseline_score"] > 0,
    "NEEDS_REVIEW",
    "MONITOR"
)


df["action"] = np.where(
    df["baseline_score"] > 0,
    "Review",
    "Monitor"
)


df[
    [
        "baseline_score",
        "reason_code",
        "action"
    ]
].head()

,baseline_score,reason_code,action
0,0,MONITOR,Monitor
1,0,MONITOR,Monitor
2,0,MONITOR,Monitor
3,0,MONITOR,Monitor
4,0,MONITOR,Monitor


In [8]:
ranked_queue = df.sort_values(
    by="baseline_score",
    ascending=False
)


ranked_queue["rank"] = range(
    1,
    len(ranked_queue)+1
)


ranked_queue.head(20)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,scroll_events,month,content_updated_date,content_created_date,days_since_last_update,ctr,baseline_score,reason_code,action,rank
15763,2026-06-02,client_23a62021009f63c4,content_526b6543f8098b5c,True,True,True,True,111,0,276,...,0,2026-06,2026-06-27,2025-08-26,-24,0.000000,50,NEEDS_REVIEW,Review,1
66170,2026-06-03,client_62f4a7e64f5e0096,content_31363e6219c75946,True,False,True,<NA>,214,1,1198,...,<NA>,2026-06,2026-07-04,2025-09-16,-31,0.004673,50,NEEDS_REVIEW,Review,2
15761,2026-06-02,client_23a62021009f63c4,content_3c8b7c066c030a2a,True,True,True,True,117,2,2385,...,0,2026-06,2026-07-04,2025-08-26,-31,0.017094,50,NEEDS_REVIEW,Review,3
66169,2026-06-03,client_62f4a7e64f5e0096,content_9221b3bf83dbdeba,True,False,True,<NA>,241,0,2109,...,<NA>,2026-06,2026-07-04,2025-09-16,-31,0.000000,50,NEEDS_REVIEW,Review,4
15758,2026-06-02,client_23a62021009f63c4,content_d1ddc152b92ce182,True,True,True,True,233,0,5418,...,0,2026-06,2026-06-15,2025-08-26,-12,0.000000,50,NEEDS_REVIEW,Review,5
15757,2026-06-02,client_23a62021009f63c4,content_e5f94277f2430121,True,True,True,True,679,1,16775,...,0,2026-06,2026-06-11,2025-08-26,-8,0.001473,50,NEEDS_REVIEW,Review,6
15754,2026-06-02,client_23a62021009f63c4,content_16e0ba6d27194380,True,True,True,False,136,0,901,...,0,2026-06,2026-06-11,2025-08-26,-8,0.000000,50,NEEDS_REVIEW,Review,7
112052,2026-06-01,client_62f4a7e64f5e0096,content_73012aef18046eb3,True,False,True,<NA>,284,2,1174,...,<NA>,2026-06,2026-06-29,2026-02-25,-26,0.007042,50,NEEDS_REVIEW,Review,8
15746,2026-06-02,client_23a62021009f63c4,content_a27b382f00aa75c6,True,True,True,True,216,0,63,...,0,2026-06,2026-06-11,2025-08-26,-8,0.000000,50,NEEDS_REVIEW,Review,9
66159,2026-06-03,client_62f4a7e64f5e0096,content_f94356c8c01a4f28,True,False,True,<NA>,386,1,1913,...,<NA>,2026-06,2026-07-04,2025-09-16,-31,0.002591,50,NEEDS_REVIEW,Review,10


In [9]:
import os

os.makedirs(
    "work/outputs",
    exist_ok=True
)


ranked_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)


print("CSV created")

CSV created


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
top20 = ranked_queue.head(20)

top20[
    [
        "rank",
        "content_hash_id",
        "baseline_score",
        "action",
        "reason_code"
    ]
]

,rank,content_hash_id,baseline_score,action,reason_code
15763,1,content_526b6543f8098b5c,50,Review,NEEDS_REVIEW
66170,2,content_31363e6219c75946,50,Review,NEEDS_REVIEW
15761,3,content_3c8b7c066c030a2a,50,Review,NEEDS_REVIEW
66169,4,content_9221b3bf83dbdeba,50,Review,NEEDS_REVIEW
15758,5,content_d1ddc152b92ce182,50,Review,NEEDS_REVIEW
15757,6,content_e5f94277f2430121,50,Review,NEEDS_REVIEW
15754,7,content_16e0ba6d27194380,50,Review,NEEDS_REVIEW
112052,8,content_73012aef18046eb3,50,Review,NEEDS_REVIEW
15746,9,content_a27b382f00aa75c6,50,Review,NEEDS_REVIEW
66159,10,content_f94356c8c01a4f28,50,Review,NEEDS_REVIEW


In [11]:
print("""Top-20 Review

The top-20 items were selected based on the baseline action score.

All top-ranked items received a score of 50 because they triggered at least one warning signal.

The ranking is directional and used for decision-support. It is not a prediction.


### Rank 1
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item triggered a warning signal from the baseline rule.
What would make it wrong: The signal may not represent a real issue and the item may still perform well.


### Rank 2
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item showed an observed condition that increased review priority.
What would make it wrong: The condition may be temporary and may recover without action.


### Rank 3
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The baseline rule identified this item as needing further inspection.
What would make it wrong: The selected threshold may be too broad and create false positives.


### Rank 4
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item matched one of the baseline warning signals.
What would make it wrong: Other factors not included in the rule may explain the result.


### Rank 5
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The score indicates that the item should be manually reviewed.
What would make it wrong: The observed change may be normal variation.


### Rank 6
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item received priority based on historical observable signals.
What would make it wrong: The rule does not include all possible context.


### Rank 7
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item crossed the baseline review threshold.
What would make it wrong: The threshold may incorrectly prioritize some items.


### Rank 8
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item was ranked because the baseline detected a warning pattern.
What would make it wrong: The pattern may not continue in future periods.


### Rank 9
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item showed a measurable signal requiring attention.
What would make it wrong: The signal alone may not indicate a necessary action.


### Rank 10
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item was prioritized using observed data signals.
What would make it wrong: A simple baseline may miss important business context.


### Rank 11
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item received a higher ranking from the baseline score.
What would make it wrong: The score does not guarantee that improvement is required.


### Rank 12
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item matched the rule conditions.
What would make it wrong: The rule may identify some normal cases as risky.


### Rank 13
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item was selected for further investigation.
What would make it wrong: Missing signals could change the priority.


### Rank 14
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The baseline observed a possible performance concern.
What would make it wrong: External changes may affect the observed metric.


### Rank 15
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item has a signal that suggests review.
What would make it wrong: The signal may not have a direct relationship with performance.


### Rank 16
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The baseline rule placed this item in the review queue.
What would make it wrong: Human review may find no meaningful issue.


### Rank 17
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item showed an observed warning signal.
What would make it wrong: Short-term changes may create a false alert.


### Rank 18
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item was prioritized by the baseline scoring system.
What would make it wrong: The simple rule may not capture the full situation.


### Rank 19
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item crossed the review threshold.
What would make it wrong: The threshold choice may not be optimal.


### Rank 20
Action: Review
Reason code: NEEDS_REVIEW
Confidence note: The item was selected based on observed signals.
What would make it wrong: Further analysis may show that no action is needed""")

Top-20 Review

The top-20 items were selected based on the baseline action score.

All top-ranked items received a score of 50 because they triggered at least one warning signal.

The ranking is directional and used for decision-support. It is not a prediction.


### Rank 1
Action: Review  
Reason code: NEEDS_REVIEW  
Confidence note: The item triggered a warning signal from the baseline rule.  
What would make it wrong: The signal may not represent a real issue and the item may still perform well.


### Rank 2
Action: Review  
Reason code: NEEDS_REVIEW  
Confidence note: The item showed an observed condition that increased review priority.  
What would make it wrong: The condition may be temporary and may recover without action.


### Rank 3
Action: Review  
Reason code: NEEDS_REVIEW  
Confidence note: The baseline rule identified this item as needing further inspection.  
What would make it wrong: The selected threshold may be too broad and create false positives.


### Rank 4
Action

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
print("""## Weak picks + leakage check

### Weak picks

Some top-ranked picks may be weak because:

- All top-20 items received the same baseline score (50), so the ranking order between them is not strongly separated.
- The baseline rule uses only a small number of signals, so it may miss other important factors.
- A detected warning signal does not always mean that an item requires action.
- Some items may be incorrectly prioritized because temporary changes can affect the observed metrics.
- Content that appears stale may still perform well and may not need updates.


### Leakage check

Confirmed:

- No product flags were used in the baseline score.
- No future performance windows were used.
- No label-derived features were used.
- The score was created only from observable historical/current signals.

The baseline is a simple decision-support rule. It is used as a benchmark that future ML models should improve upon.

""")

## Weak picks + leakage check

### Weak picks

Some top-ranked picks may be weak because:

- All top-20 items received the same baseline score (50), so the ranking order between them is not strongly separated.
- The baseline rule uses only a small number of signals, so it may miss other important factors.
- A detected warning signal does not always mean that an item requires action.
- Some items may be incorrectly prioritized because temporary changes can affect the observed metrics.
- Content that appears stale may still perform well and may not need updates.


### Leakage check

Confirmed:

- No product flags were used in the baseline score.
- No future performance windows were used.
- No label-derived features were used.
- The score was created only from observable historical/current signals.

The baseline is a simple decision-support rule. It is used as a benchmark that future ML models should improve upon.




## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.